In [49]:
import requests
import json
import pandas as pd


In [50]:
BASE = "https://gamma-api.polymarket.com/events"

def fetch_events_page(offset, limit = 100):
    response = requests.get(
        BASE,
        params={
            "closed": "true",
            "offset": offset,
            "limit": limit,
            "order": "closedTime",
            "ascending": "false"
        }
    )
    return response.json()

events = fetch_events_page(0, 5);

In [51]:
def is_resolved(market):
    x = market.get("outcomePrices")
    if x is None:
        return False
    try:
        parsed = json.loads(x)
        numbers = [float(p) for p in parsed]
    except (json.JSONDecodeError, TypeError, ValueError):
        return False
    if len(numbers) != 2:
        return False
    return (numbers[0] > 0.99 and numbers[1] < 0.01) or (numbers[0] < 0.01 and numbers[1] > 0.99)

In [52]:
resolved_markets = []
for event in events:
    for market in event["markets"]:
        if is_resolved(market):
            resolved_markets.append(market)

print(resolved_markets)

[{'id': '2073470', 'question': 'Hyperliquid Up or Down - April 25, 10:40PM-10:45PM ET', 'conditionId': '0x9e00429285fb36109592bd34f75821d37703c0654ac33fbd273337eb270f5d7d', 'slug': 'hype-updown-5m-1777171200', 'resolutionSource': 'https://data.chain.link/streams/hype-usd', 'endDate': '2026-04-26T02:45:00Z', 'liquidity': '0', 'startDate': '2026-04-25T02:48:41.797555Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/hyperliquid-16f0eafaeb.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/hyperliquid-16f0eafaeb.png', 'description': 'This market will resolve to "Up" if the Hyperliquid price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to "Down".\nThe resolution source for this market is information from Chainlink, specifically the HYPE/USD data stream available at https://data.chain.link/streams/hype-usd.\nPlease note that this market is about the price 

In [53]:
for e in events:
    print(e["title"], "|", e.get("closedTime", "no closedTime field"))

Hyperliquid Up or Down - April 25, 10:40PM-10:45PM ET | 2026-04-26T02:45:28Z
Solana Up or Down - April 25, 10:40PM-10:45PM ET | 2026-04-26T02:45:28Z
Ethereum Up or Down - April 25, 10:30PM-10:45PM ET | 2026-04-26T02:45:26Z
Dogecoin Up or Down - April 25, 10:30PM-10:45PM ET | 2026-04-26T02:45:26Z
Dogecoin Up or Down - April 25, 10:40PM-10:45PM ET | 2026-04-26T02:45:26Z


In [54]:
TOP_LEVEL_CATEGORIES = [
    "Politics",
    "Crypto",
    "Finance",
    "Sports",
    "Weather",
    "Pop Culture",
    "Science",
    "Geopolitics",
]

def classify_event(event):
    tag_labels = [t.get("label", "") for t in event.get("tags", [])]
    
    # Map noisy tag labels to clean top-level categories
    label_to_category = {
        "Politics": "Politics",
        "Crypto": "Crypto",
        "Bitcoin": "Crypto",
        "Ethereum": "Crypto",
        "Finance": "Finance",
        "Stocks": "Finance",
        "Equities": "Finance",
        "Stock Prices": "Finance",
        "Sports": "Sports",
        "NFL": "Sports",
        "NBA": "Sports",
        "Tennis": "Sports",
        "Soccer": "Sports",
        "Weather": "Weather",
        "Pop Culture": "Pop Culture",
        "Celebrities": "Pop Culture",
        "Geopolitics": "Geopolitics",
    }

    priority = ["Politics", "Crypto", "Sports", "Weather", "Pop Culture", "Geopolitics", "Finance"]

    for cat in priority:
        if any(label_to_category.get(label, "") == cat for label in tag_labels):
            return cat
    
    return "Other"

In [55]:
def extract_market_row(market, event):
    prices = json.loads(market["outcomePrices"])
    yes_price = float(prices[0])
    no_price = float(prices[1])

    return {
        "market_id": market.get("id", ""),
        "question": market.get("question", ""),
        "category": classify_event(event),
        "yes_resolved": 1 if yes_price > 0.99 else 0,
        "created_at": market.get("createdAt", None),
        "closed_time": event.get("closedTime", None),
        "end_date": market.get("endDate", None),
        "volume": float(market.get("volume", 0)),
        "volume24hr": float(market.get("volume24hr", 0)),
        "volume1wk": float(market.get("volume1wk", 0)),
        "volume1mo": float(market.get("volume1mo", 0)),
        "last_trade_price": float(market.get("lastTradePrice", 0)),
        "one_day_price_change": float(market.get("oneDayPriceChange", 0)),
        "one_week_price_change": float(market.get("oneWeekPriceChange", 0)),
        "one_month_price_change": float(market.get("oneMonthPriceChange", 0))
    }

In [56]:
all_rows = []
offset = 0
PAGE_SIZE = 100
TARGET_RESOLVED = 5000
MAX_OFFSET = 10000
CUTOFF = "2026-04-24T00:00:00Z"

while len(all_rows) < TARGET_RESOLVED:
    if offset > MAX_OFFSET:
        print("Hit max offset limit, stopping")
        break
    
    events = fetch_events_page(offset, PAGE_SIZE)
    if not events:
        print("No more events returned, stopping")
        break

    for event in events:
        for market in event.get("markets", []):
            closed_time = market.get("closedTime")
            if closed_time is not None and closed_time > CUTOFF:
                continue
            
            if is_resolved(market):
                all_rows.append(extract_market_row(market, event))
    offset += PAGE_SIZE
    print(f"Offset {offset}: have {len(all_rows)} resolved markets")

df = pd.DataFrame(all_rows)
print()
print("Shape:", df.shape)
print()
print("Categories:")
print(df["category"].value_counts())

Offset 100: have 0 resolved markets
Offset 200: have 1 resolved markets
Offset 300: have 1 resolved markets
Offset 400: have 1 resolved markets
Offset 500: have 1 resolved markets
Offset 600: have 1 resolved markets
Offset 700: have 2 resolved markets
Offset 800: have 2 resolved markets
Offset 900: have 2 resolved markets
Offset 1000: have 2 resolved markets
Offset 1100: have 23 resolved markets
Offset 1200: have 30 resolved markets
Offset 1300: have 32 resolved markets
Offset 1400: have 32 resolved markets
Offset 1500: have 32 resolved markets
Offset 1600: have 32 resolved markets
Offset 1700: have 49 resolved markets
Offset 1800: have 52 resolved markets
Offset 1900: have 52 resolved markets
Offset 2000: have 52 resolved markets
Offset 2100: have 53 resolved markets
Offset 2200: have 53 resolved markets
Offset 2300: have 53 resolved markets
Offset 2400: have 53 resolved markets
Offset 2500: have 53 resolved markets
Offset 2600: have 53 resolved markets
Offset 2700: have 76 resolved m

In [57]:
df.to_csv("resolved_markets.csv", index=False)

In [62]:
df['volume'].describe()
df['volume'].quantile([0.05, 0.10, 0.25, 0.50])

0.05      0.000000
0.10      5.580000
0.25    183.810442
0.50    820.901580
Name: volume, dtype: float64

In [59]:
for thresh in [0, 50, 100, 250, 500, 1000]:
    surviving = df[df['volume'] >= thresh]
    print(f"\nThreshold ${thresh}: {len(surviving)} markets total")
    print(surviving['category'].value_counts())


Threshold $0: 5076 markets total
category
Crypto         2191
Sports         1861
Weather         532
Finance         161
Geopolitics     123
Politics        115
Other            93
Name: count, dtype: int64

Threshold $50: 4296 markets total
category
Crypto         2150
Sports         1248
Weather         532
Finance         140
Politics        113
Other            57
Geopolitics      56
Name: count, dtype: int64

Threshold $100: 4107 markets total
category
Crypto         2118
Sports         1095
Weather         532
Finance         136
Politics        113
Other            57
Geopolitics      56
Name: count, dtype: int64

Threshold $250: 3550 markets total
category
Crypto         1897
Sports          783
Weather         532
Politics        113
Finance         112
Other            57
Geopolitics      56
Name: count, dtype: int64

Threshold $500: 2976 markets total
category
Crypto         1494
Sports          651
Weather         525
Politics        111
Finance          87
Geopolitics   